In [1]:
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224

# Diviser les données en 80% train et 20% pour validation/test
train_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds) - val_size  # Reste pour test

val_ds = val_test_ds.take(val_size)  # Premier 50% pour validation
test_ds = val_test_ds.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds)}")
print(f"Nombre de batches dans val_ds: {len(val_ds)}")
print(f"Nombre de batches dans test_ds: {len(test_ds)}")

Found 1968 files belonging to 3 classes.
Using 1575 files for training.
Found 1968 files belonging to 3 classes.
Using 393 files for validation.
Nombre de batches dans train_ds: 50
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [2]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds_maiis))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds_maiis) - val_size  # Reste pour test

val_ds_maiis = val_test_ds_maiis.take(val_size)  # Premier 50% pour validation
test_ds_maiis = val_test_ds_maiis.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_maiis)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_maiis)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_maiis)}")

Found 1930 files belonging to 3 classes.
Using 1544 files for training.
Found 1930 files belonging to 3 classes.
Using 386 files for validation.
Nombre de batches dans train_ds: 49
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [3]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size_mixte = int(0.5 * len(val_test_ds_mixte))  # 50% de val_test_ds pour validation
test_size_mixte = len(val_test_ds_mixte) - val_size_mixte  # Reste pour test

val_ds_mixte = val_test_ds_maiis.take(val_size_mixte)  # Premier 50% pour validation
test_ds_mixte = val_test_ds_maiis.skip(val_size_mixte)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_mixte)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_mixte)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_mixte)}")

Found 1940 files belonging to 3 classes.
Using 1552 files for training.
Found 1940 files belonging to 3 classes.
Using 388 files for validation.
Nombre de batches dans train_ds: 49
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [4]:
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import time

# Définir la hauteur et la largeur des images
img_height = 224  # Taille d'entrée requise pour InceptionResNetV2
img_width = 224

# Charger le modèle InceptionResNetV2
base_model = InceptionResNetV2(input_shape=(img_height, img_width, 3),
                               include_top=False,
                               weights='imagenet')
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

for layer in base_model.layers[-50:]:
    layer.trainable = True

# Ajouter une couche Global Average Pooling
global_average_layer = layers.GlobalAveragePooling2D()(base_model.output)

# Ajouter les nouvelles couches demandées
dense_1 = layers.Dense(1024, activation='relu')(global_average_layer)
dropout_1 = layers.Dropout(0.5)(dense_1)

dense_2 = layers.Dense(512, activation='relu')(dropout_1)
dropout_2 = layers.Dropout(0.5)(dense_2)

dense_3 = layers.Dense(256, activation='relu')(dropout_2)
dropout_3 = layers.Dropout(0.5)(dense_3)

dense_4 = layers.Dense(128, activation='relu')(dropout_3)
dropout_4 = layers.Dropout(0.5)(dense_4)

# Créer des sorties pour chaque nutriment
outputs = []
for nutrient in range(13):
    output = layers.Dense(3, activation='softmax', name=f'nutrient_{nutrient}')(dropout_4)
    outputs.append(output)

# Créer le modèle final
model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilation du modèle avec précision et rappel pour chaque sortie
metrics = ['accuracy', Precision(name='precision'), Recall(name='recall')]
metrics_list = [metrics] * 13

model.compile(optimizer='adam',
              loss=['categorical_crossentropy'] * 13,
              metrics=metrics_list)

In [5]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir le checkpoint pour sauvegarder les meilleurs poids
checkpoint = ModelCheckpoint('Inception_V4_weights.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             mode='max',
                             save_best_only=True)

# Early stopping pour arrêter l'entraînement si la validation stagne
early = EarlyStopping(monitor="val_loss",
                      mode="min",
                      restore_best_weights=True,
                      patience=5)

# Liste des callbacks
callbacks_list = [checkpoint, early]

In [6]:
import time
# Entraînement du modèle tout en mesurant le temps
start_time = time.time()

history = model.fit(
    train_val_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks_list,
    verbose=True,
    shuffle=True
)

end_time = time.time()

# Afficher le temps d'entraînement
training_time = end_time - start_time
print(f"Temps d'apprentissage : {training_time} secondes")

# Calcul manuel du F1-score après l'entraînement
precision = history.history['precision'][-1]
recall = history.history['recall'][-1]
f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon()) 
print(f'F1 Score: {f1:.4f}')

Epoch 1/15


C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\optimizers\base_optimizer.py:678: UserWarning: Gradients do not exist for variables ['kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - loss: 1.8092 - nutrient_0_accuracy: 0.3476 - nutrient_0_precision: 0.3576 - nutrient_0_recall: 0.2373 

C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\callbacks\model_checkpoint.py:206: UserWarning: Can save best model only with val_accuracy available, skipping.
  self._save_model(epoch=epoch, batch=None, logs=logs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 705s 12s/step - loss: 1.8080 - nutrient_0_accuracy: 0.3475 - nutrient_0_precision: 0.3577 - nutrient_0_recall: 0.2372 - val_loss: 1.4348 - val_nutrient_0_accuracy: 0.2917 - val_nutrient_0_precision: 0.2642 - val_nutrient_0_recall: 0.2188
Epoch 2/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 510s 10s/step - loss: 1.2747 - nutrient_0_accuracy: 0.3773 - nutrient_0_precision: 0.3983 - nutrient_0_recall: 0.1472 - val_loss: 1.0933 - val_nutrient_0_accuracy: 0.4583 - val_nutrient_0_precision: 0.0000e+00 - val_nutrient_0_recall: 0.0000e+00
Epoch 3/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 547s 11s/step - loss: 1.1083 - nutrient_0_accuracy: 0.3741 - nutrient_0_precision: 0.5203 - nutrient_0_recall: 0.0845 - val_loss: 1.0857 - val_nutrient_0_accuracy: 0.4531 - val_nutrient_0_precision: 0.0000e+00 - val_nutrient_0_recall: 0.0000e+00
Epoch 4/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 548s 11s/step - loss: 1.0779 - nutrient_0_accuracy: 0.3878 - nutrient_0_precision: 0.6239 - nutrient_0_recall: 0.0834 - val_loss:

KeyError: 'precision'

In [7]:
model.save("model/InceptionV4_04_11.h5")

In [8]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 33s 4s/step - loss: 1.0899 - nutrient_0_accuracy: 0.4200 - nutrient_0_precision: 0.0000e+00 - nutrient_0_recall: 0.0000e+00
Nombre total de résultats: 4
Résultats de l'évaluation: [1.0898081064224243, 0.43283581733703613, 0.0, 0.0]
La structure des résultats est différente de celle attendue.


In [9]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_mixte)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 35s 4s/step - loss: 1.0758 - nutrient_0_accuracy: 0.4406 - nutrient_0_precision: 0.0000e+00 - nutrient_0_recall: 0.0000e+00
Nombre total de résultats: 4
Résultats de l'évaluation: [1.0779223442077637, 0.4639175236225128, 0.0, 0.0]
La structure des résultats est différente de celle attendue.


In [10]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_maiis)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - loss: 1.0729 - nutrient_0_accuracy: 0.4607 - nutrient_0_precision: 0.0000e+00 - nutrient_0_recall: 0.0000e+00
Nombre total de résultats: 4
Résultats de l'évaluation: [1.071969747543335, 0.47422680258750916, 0.0, 0.0]
La structure des résultats est différente de celle attendue.


In [11]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

6/6 ━━━━━━━━━━━━━━━━━━━━ 29s 5s/step - loss: 1.0812 - nutrient_0_accuracy: 0.4794 - nutrient_0_precision: 0.0000e+00 - nutrient_0_recall: 0.0000e+00
Nombre total de résultats: 4
Résultats de l'évaluation: [1.086136817932129, 0.421875, 0.0, 0.0]
La structure des résultats est différente de celle attendue.


In [12]:
import time
from tensorflow.keras import backend as K

# Fonction pour calculer la moyenne d'une métrique
def mean_metric(metric_values):
    return sum(metric_values) / len(metric_values)

# Nombre de nutriments
num_nutrients = 13

# Fonction pour calculer les moyennes pour chaque nutriment
def mean_nutrient_metric(metric_name, history):
    metrics = []
    for nutrient in range(num_nutrients):
        key = f'nutrient_{nutrient}_{metric_name}'
        if key in history:
            metrics.append(history[key])
    return [mean_metric(metric) for metric in zip(*metrics)]  # Moyenne sur les époques

# Calcul des moyennes pour l'ensemble des époques (entraînement)
mean_train_loss = mean_metric(history.history['loss'])
mean_train_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_train_precision = mean_nutrient_metric('precision', history.history)
mean_train_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (entraînement)
f1_train_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                   for p, r in zip(mean_train_precision, mean_train_recall)]
mean_f1_train = mean_metric(f1_train_scores)

# Calcul des moyennes pour l'ensemble des époques (validation)
mean_val_loss = mean_metric(history.history['val_loss'])
mean_val_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_val_precision = mean_nutrient_metric('precision', history.history)
mean_val_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (validation)
f1_val_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                 for p, r in zip(mean_val_precision, mean_val_recall)]
mean_f1_val = mean_metric(f1_val_scores)

# Affichage des résultats
print(f"Moyenne de la perte sur l'ensemble d'entraînement : {mean_train_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble d'entraînement : {mean_metric(mean_train_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble d'entraînement : {mean_metric(mean_train_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble d'entraînement : {mean_metric(mean_train_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble d'entraînement : {mean_f1_train:.4f}")

print(f"Moyenne de la perte sur l'ensemble de validation : {mean_val_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble de validation : {mean_metric(mean_val_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble de validation : {mean_metric(mean_val_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble de validation : {mean_metric(mean_val_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble de validation : {mean_f1_val:.4f}")

Moyenne de la perte sur l'ensemble d'entraînement : 1.1846
Moyenne de l'accuracy sur l'ensemble d'entraînement : 0.3913
Moyenne de la précision sur l'ensemble d'entraînement : 0.5793
Moyenne du rappel sur l'ensemble d'entraînement : 0.0976
Moyenne du F1-score sur l'ensemble d'entraînement : 0.1549
Moyenne de la perte sur l'ensemble de validation : 1.1807
Moyenne de l'accuracy sur l'ensemble de validation : 0.3913
Moyenne de la précision sur l'ensemble de validation : 0.5793
Moyenne du rappel sur l'ensemble de validation : 0.0976
Moyenne du F1-score sur l'ensemble de validation : 0.1549
